<div align="center">

# Data Projects and Hackathon 3  
## Project 
Sergio Fernandez, Alessandro Mecchia 

</div>

In [21]:
import pandas as pd
import json
import json
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import seaborn as sns
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
import numpy as np
from transformers import pipeline
import requests
import time
from pathlib import Path
import subprocess
import sys
import time

try:
    import pyarrow as pa
    import pyarrow.json as paj
    import pyarrow.parquet as pq
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyarrow"])
    import pyarrow as pa
    import pyarrow.json as paj
    import pyarrow.parquet as pq
    
import pyarrow.parquet as pq

import pyarrow.parquet as pq
import ipywidgets as widgets
from IPython.display import display


## Functions 

In [22]:
def pct(x): 
    return f"{x:,} ({x/n*100:.1f}%)"

def is_empty(val):
    if val is None:
        return True
    if isinstance(val, str):
        return val == ""
    if isinstance(val, (list, np.ndarray)):
        return len(val) == 0
    return False

In [23]:
def get_lang_from_openalex(doi):
    if not doi or doi == "":
        return None
    
    url = f"https://api.openalex.org/works/https://doi.org/{doi}"
    headers = {"User-Agent": "DBLP-lang-imputation/1.0 (your@email.com)"}
    
    try:
        r = requests.get(url, headers=headers, timeout=10)
        if r.status_code == 200:
            data = r.json()
            return data.get("language", None) 
        elif r.status_code == 429:
            time.sleep(5)   
            return get_lang_from_openalex(doi)
    except Exception:
        return None
    return None

In [24]:
import json

def impute_with_cache(df, col, fetch_fn, cache_path):
    cache_path = Path(cache_path)
    mask = df[col].apply(is_empty) & ~df["doi"].apply(is_empty)
    missing_df = df[mask]

    if cache_path.exists():
        with open(cache_path, "r") as f:
            recovered = {int(k): v for k, v in json.load(f).items()}
        print(f"Cache found: loaded {len(recovered)} values from {cache_path}")
    else:
        recovered = {}
        for i, (idx, row) in enumerate(missing_df.iterrows()):
            value = fetch_fn(row["doi"])
            if value:
                recovered[idx] = value

            if (i + 1) % 10 == 0:
                print(f"  [{i+1}/{len(missing_df)}] recovered until now: {len(recovered)}")

            time.sleep(0.1)

        with open(cache_path, "w") as f:
            json.dump(recovered, f)
        print(f"Cache saved to {cache_path}")

    # Assign row by row to avoid issues with list values
    for idx, value in recovered.items():
        df.at[idx, col] = value

    print(f"Imputation completed:")
    print(f"Missing '{col}' before: {mask.sum()} and after: {df[col].apply(is_empty).sum()}")

In [25]:
import json

def impute_with_cache(df, col, fetch_fn, cache_path):
    cache_path = Path(cache_path)
    mask = df[col].apply(is_empty) & ~df["doi"].apply(is_empty)
    missing_df = df[mask]

    if cache_path.exists():
        with open(cache_path, "r") as f:
            recovered = {int(k): v for k, v in json.load(f).items()}
        print(f"Cache found: loaded {len(recovered)} values from {cache_path}")
    else:
        recovered = {}
        for i, (idx, row) in enumerate(missing_df.iterrows()):
            value = fetch_fn(row["doi"])
            if value:
                recovered[idx] = value

            if (i + 1) % 10 == 0:
                print(f"  [{i+1}/{len(missing_df)}] recovered until now: {len(recovered)}")

            time.sleep(0.1)

        with open(cache_path, "w") as f:
            json.dump(recovered, f)
        print(f"Cache saved to {cache_path}")

    for idx, value in recovered.items():
        df.at[idx, col] = value

    print(f"Imputation completed:")
    print(f"Missing '{col}' before: {mask.sum()} and after: {df[col].apply(is_empty).sum()}")

In [26]:
def get_lang_from_openalex(doi):
    if not doi or doi == "":
        return None
    try:
        r = requests.get(
            f"https://api.openalex.org/works/https://doi.org/{doi}",
            headers={"User-Agent": "DBLP-imputation/1.0"},
            timeout=10
        )
        if r.status_code != 200:
            return None
        return r.json().get("language", None)
    except Exception:
        return None


def get_keywords_from_openalex(doi):
    if not doi or doi == "":
        return None
    try:
        r = requests.get(
            f"https://api.openalex.org/works/https://doi.org/{doi}",
            headers={"User-Agent": "DBLP-imputation/1.0"},
            timeout=10
        )
        if r.status_code != 200:
            return None
        concepts = r.json().get("concepts", [])
        keywords = [
            c["display_name"] for c in concepts
            if c.get("level", 0) >= 1 and c.get("score", 0) >= 0.3
        ]
        return keywords if keywords else None
    except Exception:
        return None

## Data Creation 

da eseguire una sola volta 

In [27]:
DATA_DIR = Path("data")
SOURCE_PATH = DATA_DIR / "DBLP-Citation-network-V18.jsonl"
TARGET_PATH = DATA_DIR / "DBLP-Citation-network-V18.parquet"
BLOCK_SIZE = 64 * 1024 * 1024  

if not SOURCE_PATH.exists():
    raise FileNotFoundError(f"File non trovato: {SOURCE_PATH}")

if pa.Codec.is_available("zstd"):
    COMPRESSION = "zstd"
elif pa.Codec.is_available("snappy"):
    COMPRESSION = "snappy"
else:
    COMPRESSION = None

print(f"Input : {SOURCE_PATH} ({SOURCE_PATH.stat().st_size / 1024**3:.2f} GiB)")
print(f"Output: {TARGET_PATH}")
print(f"Compressione: {COMPRESSION}")
print(f"Block size: {BLOCK_SIZE / 1024**2:.0f} MiB")

Input : data\DBLP-Citation-network-V18.jsonl (14.85 GiB)
Output: data\DBLP-Citation-network-V18.parquet
Compressione: zstd
Block size: 64 MiB


In [28]:
if TARGET_PATH.exists():
    print(f"Parquet already exists, skipping conversion.")
else:
    reader = paj.open_json(
        SOURCE_PATH,
        read_options=paj.ReadOptions(block_size=BLOCK_SIZE),
    )

    writer = None
    rows_written = 0
    batches_written = 0
    started_at = time.perf_counter()

    try:
        while True:
            try:
                batch = reader.read_next_batch()
            except StopIteration:
                break

            if writer is None:
                writer = pq.ParquetWriter(
                    TARGET_PATH,
                    batch.schema,
                    compression=COMPRESSION,
                )

            writer.write_batch(batch)
            rows_written += batch.num_rows
            batches_written += 1

            if batches_written % 25 == 0:
                elapsed = time.perf_counter() - started_at
                print(f"Batch: {batches_written:>5} | Rows: {rows_written:>12,} | Elapsed: {elapsed:>8.1f}s")

        if writer is None:
            raise RuntimeError("JSONL file seems empty: no batch read.")
    finally:
        reader.close()
        if writer is not None:
            writer.close()

    elapsed = time.perf_counter() - started_at
    print(f"Conversion completed in {elapsed:.1f}s")
    print(f"Rows written : {rows_written:,}")
    print(f"JSONL size   : {SOURCE_PATH.stat().st_size / 1024**3:.2f} GiB")
    print(f"Parquet size : {TARGET_PATH.stat().st_size / 1024**3:.2f} GiB")

Parquet already exists, skipping conversion.


## Data Exploration 

In [29]:
pf = pq.ParquetFile(r"data\DBLP-Citation-network-V18.parquet")

print(f"Total rows   : {pf.metadata.num_rows:,}")
print(f"Columns      : {pf.metadata.num_columns}")
print(f"Row groups   : {pf.metadata.num_row_groups}")

Total rows   : 6,729,828
Columns      : 21
Row groups   : 238


In [30]:
for i, name in enumerate(pf.schema_arrow.names):
    print(f"{i}. {name}")

0. id
1. title
2. abstract
3. keywords
4. year
5. authors
6. references
7. page_start
8. page_end
9. lang
10. volume
11. issue
12. issn
13. isbn
14. doi
15. url
16. n_citation
17. venue
18. doc_type


In [31]:
pf = pq.ParquetFile(r"data\DBLP-Citation-network-V18.parquet")
columns = pf.schema_arrow.names

dropdown = widgets.Dropdown(options=columns, description="Colonna:")
output = widgets.Output()

def on_change(change):
    if change["type"] == "change" and change["name"] == "value":
        with output:
            output.clear_output()
            batch = next(pf.iter_batches(batch_size=10, columns=[change["new"]]))
            df = batch.to_pandas()
            display(df[change["new"]])

dropdown.observe(on_change)
display(dropdown, output)

Dropdown(description='Colonna:', options=('id', 'title', 'abstract', 'keywords', 'year', 'authors', 'reference…

Output()

In [32]:
batch = next(pf.iter_batches(batch_size=1))
paper = batch.to_pandas().iloc[0]

for col, val in paper.items():
    print(f"{col:15}: {val}")

id             : 5390877920f70186a0d2ce7f
title          : Top-Down Construction of 3-D Mechanical Object Shapes from Engineering Drawings
abstract       : 
keywords       : ['First Page' '3-D Mechanical Object Shapes' 'Engineering Drawings'
 'Top-Down Construction']
year           : 1984
authors        : [{'id': '62aad3b2d9f2040d085dc9f9', 'name': 'H YOSHIURA', 'org': 'Hitachi Research Laboratory, Hitachi Ltd.'}
 {'id': '5608ec4a45cedb3396db2920', 'name': 'K FUJIMURA', 'org': 'Univ. of Tokyo, Tokyo, Japan'}
 {'id': '548a62d3dabfae8a11fb49e2', 'name': 'TL KUNII', 'org': None}]
references     : ['5390962020f70186a0df3bf2' '5390879d20f70186a0d43d74'
 '5390a1e620f70186a0e59c05' '5390b1d220f70186a0ee1bd6'
 '53909f8220f70186a0e3cfc7' '5390962020f70186a0df3f87'
 '53e9b8a8b7602d970447e86b']
page_start     : 32
page_end       : 40
lang           : en
volume         : 17
issue          : 12
issn           : 0018-9162
isbn           : 
doi            : 10.1109/mc.1984.1659026
url            : []

### Missing values 

In [33]:
batch = next(pf.iter_batches(batch_size=10_000))
df = batch.to_pandas()
n = len(df)

# Simple columns
for col in ["id", "title", "abstract", "year", "page_start", "page_end",
            "lang", "volume", "issue", "issn", "isbn", "doi", "venue", "doc_type"]:
    missing = df[col].apply(is_empty).sum()
    print(f"  {col:15}: {pct(missing)}")

# Lists
print()
for col in ["keywords", "references", "url"]:
    missing = df[col].apply(is_empty).sum()
    print(f"  {col:15}: {pct(missing)}")

# Authors 
print()
authors_flat = pd.DataFrame(df["authors"].explode().dropna().tolist())
total_authors = len(authors_flat)
for col in ["id", "name", "org"]:
    missing = authors_flat[col].apply(is_empty).sum()
    print(f"  authors.{col:10}: {missing:,} ({missing/total_authors*100:.1f}% of autors)")

  id             : 0 (0.0%)
  title          : 0 (0.0%)
  abstract       : 516 (5.2%)
  year           : 0 (0.0%)
  page_start     : 2,318 (23.2%)
  page_end       : 2,460 (24.6%)
  lang           : 292 (2.9%)
  volume         : 4,764 (47.6%)
  issue          : 6,610 (66.1%)
  issn           : 4,710 (47.1%)
  isbn           : 9,847 (98.5%)
  doi            : 875 (8.8%)
  venue          : 56 (0.6%)
  doc_type       : 15 (0.1%)

  keywords       : 1,516 (15.2%)
  references     : 1,668 (16.7%)
  url            : 946 (9.5%)

  authors.id        : 2,789 (8.6% of autors)
  authors.name      : 0 (0.0% of autors)
  authors.org       : 5,990 (18.5% of autors)


spiegare che ci interessa imputare solo le variabili che pensiamo possano essere utili per predirre il numero di citazioni o se un paper cita un'altro.

#### 1. Lang emputation

spiegare che se avessivo avuto accesso ha una parte del paper avremmo potuto usare un modello dei trasformer per analizzare la scrittura del paper e imputare in modo corretto la lingua del paper, siccome non è il caso possiamo provare altri metodi. 

Prima di tutto dobbiamo vedere le possibilità o comunque i valori che abbiamo presenti, in caso fossero tutti inglesi allora li mettiamo inglesi tutti. 

In [34]:
lang_counts = df["lang"].value_counts(dropna=False)
lang_counts.index = lang_counts.index.fillna("(null)")
print(f"Unique values of 'lang' (sample of {n:,} rows):\n")
print(lang_counts.to_string())

Unique values of 'lang' (sample of 10,000 rows):

lang
en    9653
       292
de      41
fr       9
pt       2
zh       2
da       1


siccome non è il caso e abbiamo diverse possibilità, la prossima cosa che possiamo fare è utilizzare il doi e cercare in un'altra libbreria se riusciamo a identificare la lingua del paper

In [35]:
impute_with_cache(df, "lang",     get_lang_from_openalex,     "data/cache_lang.json")

Cache found: loaded 212 values from data\cache_lang.json
Imputation completed:
Missing 'lang' before: 214 and after: 80


abbiamo ancora 80 valori mancanti, la prossima possibilità è quella di usare l'abstract e con un modello trasnformer rilevare che lingua è e imputarla, ma prima di tutto dobbiamo vedere se per caso la lingua del abstract non è la stessa a quella inseria in lang

In [36]:
sample_by_lang = (
    df[~df["abstract"].apply(is_empty) & ~df["lang"].apply(is_empty)]
    .groupby("lang")
    .apply(lambda x: x.sample(1, random_state=42))
    .reset_index(drop=True)
    [["lang", "title", "abstract"]]
    .sort_values("lang")
)

for _, row in sample_by_lang.iterrows():
    print(f"{'─'*60}")
    print(f"LANG : {row['lang']}")
    print(f"TITLE: {row['title']}")
    print(f"ABSTRACT: {row['abstract'][:300]}...")
    print()

────────────────────────────────────────────────────────────
LANG : de
TITLE: Divergierende Urheberrechtliche Und Äußerungsrechtliche Haftung Bei Online-Archiven?
ABSTRACT: Karten am 6.11.2009 – ausweislich der von der Beklagten erteilten Gutschrift sind die ersten Freischaltungen am 3.8.2009 erfolgt – ist der Provisionsanspruch gem. § 87a Abs. 3 Satz 2 HGB insgesamt entfallen. Von einer Teilausführung des Geschäfts durch TM., die möglicherweise zu einem Teilanspruch a...

────────────────────────────────────────────────────────────
LANG : en
TITLE: A Privacy-aware Graph-based Access Control System for Healthcare Domain.
ABSTRACT: The growing concern for the protection of personal information has made it critical to implement effective technologies for privacy and data management. By observing the limitations of existing approaches, we found that there is an urgent need for a flexible, privacy-aware system that is able to mee...

────────────────────────────────────────────────────────

C:\Users\Sergio\AppData\Local\Temp\ipykernel_23000\2653525852.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(1, random_state=42))


ora che abbiamo visto che è fattibile applichiamo il metodo con il modello

In [37]:
from transformers import pipeline

lang_detector = pipeline(
    "text-classification",
    model="papluca/xlm-roberta-base-language-detection"
)

mask = df["lang"].apply(is_empty) & ~df["abstract"].apply(is_empty)

df.loc[mask, "lang"] = df.loc[mask, "abstract"].apply(
    lambda x: lang_detector(x[:512], truncation=True)[0]["label"]
)

print(f"Imputed: {mask.sum()} | Still missing: {df['lang'].apply(is_empty).sum()}")

Device set to use cuda:0


Imputed: 30 | Still missing: 50


come possiamo notare rimangono ancora 50 valori mancanti dovuto al fatto che molti non hanno l'abstract, ma comunque siamo risuciti a imputare un totale di 242 su 292 

#### 2. Keywords imputation

come primo tentativo possiamo vedere se con il doi e usando onealex possiamo ottenere delle keywords 

In [38]:
impute_with_cache(df, "keywords", get_keywords_from_openalex, "data/cache_keywords.json")

Cache found: loaded 952 values from data\cache_keywords.json
Imputation completed:
Missing 'keywords' before: 1011 and after: 564


siamo riusciti ad imputare un totale di circa 500 valori, i rimanenti sono dovuti a richieste che ritornano un errore, o a mancanti DOI, per il resto possiamo generarli partendo dall'absatract

In [39]:
kw_extractor = pipeline(
    "text2text-generation",
    model="fabiochiu/t5-base-tag-generation"
)

def extract_keywords_from_abstract(abstract):
    if not abstract or abstract == "":
        return None
    result = kw_extractor(abstract[:512], max_new_tokens=50)
    keywords = [kw.strip() for kw in result[0]["generated_text"].split(",") if kw.strip()]
    return keywords if keywords else None


mask = df["keywords"].apply(is_empty) & ~df["abstract"].apply(is_empty)
print(f"Records to impute with transformer: {mask.sum()}")

for idx, row in df[mask].iterrows():
    keywords = extract_keywords_from_abstract(row["abstract"])
    if keywords:
        df.at[idx, "keywords"] = keywords

print(f"Still missing: {df['keywords'].apply(is_empty).sum()}")

Device set to use cuda:0


Records to impute with transformer: 328
Still missing: 236


## Data Visualizzation 